# NB4 · Justifying the decision

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What this notebook does

The model produces a probability but does not say why. Showing a clinician a number alone
is not enough; the reasoning is needed too.

There is a legal counterpart as well. As set out in the lecture on 16 September, whether
decision support software counts as a medical device turns on whether the health care
professional can independently review the basis of the recommendation. The layer added
here is that criterion expressed in software.

You will produce reasoning at two levels: what the model looks at in general, and what
drove the decision for one particular patient.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Code carried from the previous notebook

Paste the whole block collected at the end of the previous notebook into the cell
below. Do not delete the `#@cdss` marker on the first line; that block is collected
again at the end of this notebook and carried to the next one.

Running the block rebuilds everything you wrote in the earlier notebooks. Where it
reads data from the web, the cell may take a few seconds.


In [ ]:
#@cdss onceki_defter
# Paste the generated code below this line.


### Check · The carried code


In [ ]:
kit.check_defined('model', 'X_train', 'X_test', 'y_test', 'probability', 'result')


---

## Step 1 · What the model looks at in general

The first question is which pieces of information the model leans on when it decides.

The way to measure this is simple. One piece of information is shuffled, so its values
swap places among patients, and we observe how far performance falls. A large fall means
the model depends on it.

In the result, the spread matters as much as the mean. On a cohort of one hundred
patients, the difference between repeats can exceed the difference between neighbouring
pieces of information. Where that happens the ranking itself is unreliable and cannot be
presented as a finding.


### Prompt 1

```
Write a piece of work that measures which information the model depends on. Name it
overall_reasoning.

For each piece of information do this: shuffle its values among the patients at random
and observe how far the performance of the model falls. Repeat the shuffling several
times so we can also see how much the result moves.

Give the result back as a table containing:
  feature -> the name of the information
  effect  -> the average fall in performance
  spread  -> the variation across repeats
  stable  -> true where the effect exceeds twice the spread, false otherwise

Sort the table by effect, largest first.

Then run it on the test group, keep the result under the name reasoning_table and show
the first fifteen rows.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named overall_reasoning that returns a table.
There must be a table named reasoning_table containing feature, effect, spread and
stable.
```


In [ ]:
#@cdss genel_gerekce
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_function('overall_reasoning')
kit.check_frame(reasoning_table, name='reasoning_table',
                required=['feature', 'effect', 'spread', 'stable'])


In [ ]:
unstable = int((~reasoning_table['stable'].astype(bool)).sum())
print(f'Items whose ranking is not reliable: {unstable} / {len(reasoning_table)}')
print('Where this count is high, do not present the ordering as a finding.')


### Python note · Loops and lists

In the code you will see lines of the form `for feature in ...:`. This is a **loop**: it
repeats the same work for every item in a list. Here the shuffling is repeated for each
piece of information.

The lines below a loop are indented, and that is not decoration. In Python, indentation
determines which lines belong to the loop; it takes the place of the braces used in other
languages.

A **list** is an ordered collection, such as `[0.12, 0.08, 0.31]`. Unlike a dictionary it
is accessed by position rather than by name. The results of the shuffling repeats are
gathered in a list first, then their mean and spread are computed.


---

## Step 2 · What drove the decision for one patient

Overall reasoning describes the model; per patient reasoning describes why this patient
reached your attention. The clinician needs the second.

In this step you will look at three patients: one the model caught correctly, one it
alerted on without cause, and one it missed.

**The one alerted on without cause matters most.** Reasoning that makes a wrong
prediction look sensible is where explanation launders error. Seeing it once teaches more
than reading the definition of the concept.


### Prompt 2

```
Write a piece of work that produces the reasoning behind the model's decision for a
single patient. Name it patient_reasoning and let it take one patient's information.

Compute which pieces of information pushed the decision up and which pushed it down for
this patient. Give each contribution together with the actual value for that patient.

Give the result back as a table with the columns feature, value and contribution, sorted
by the size of the contribution.

Also write a short piece of code that finds three patients: one the model caught
correctly, one it alerted on without cause, and one it missed. Keep them in a dictionary
named cases with the keys true_alert, false_alert and missed, whose values are row
numbers. Where a case is not found, the value must be None.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named patient_reasoning that returns a table.
There must be a dictionary named cases containing true_alert, false_alert and missed.
```


In [ ]:
#@cdss hasta_gerekcesi
# Paste the generated code below this line.


### Check 2


In [ ]:
kit.check_function('patient_reasoning')

for name in ['true_alert', 'false_alert', 'missed']:
    v = cases.get(name)
    print(f'{name:<13}', 'not found' if v is None else f'row {v}')


### Look at the three cases

The supplied cell below shows the reasoning for the three patients one after another. Run
it, then answer the questions below.


In [ ]:
for name in ['true_alert', 'false_alert', 'missed']:
    row = cases.get(name)
    if row is None:
        continue
    print('=' * 60)
    print(name.upper(), '· probability', round(float(probability[row]), 3))
    print('=' * 60)
    print(patient_reasoning(X_test[row:row+1]).head(8).to_string(index=False))
    print()


### Critique of the reasoning

Answer three questions yourself, then put the same questions to the AI tool and compare.

**Could one of the high contribution items come from how the record was made rather than
from a clinical signal?** Every data type has its own version of this. In routine hospital
data the measurement count is a candidate; a patient with many measurements may simply
have been treated as severe from the outset. In medical imaging the marker a device prints
into the frame, or a hospital stamp, does the same work. In a physiological signal it is
the noise pattern of the recording device, and in clinical text the templated sentence
repeated in every note. None of the four is a clinical finding; each is a trace of the
care process.

**Would a clinician reading the reasoning for the patient alerted on without cause be
persuaded?** If so, that is a problem rather than a success; the system can defend its
wrong decision convincingly.

**Which of these contributions could be mistaken for causation?** A high contribution from
a measurement does not mean that changing it would change the outcome. Reasoning shows an
association, not a cause.


---

## End of notebook · Collecting the code

The cell below collects the code you carried from the earlier notebooks together with
what you added here, as one block. Copy the whole block; you will paste it into the
first cell of NB5.

The block is also saved as `cdss_nb4.py`. That file disappears when the Colab session closes,
so keep a copy in a text file on your own computer as well.


In [ ]:
code_so_far = kit.export(save_as='cdss_nb4.py')


## What this notebook did

Two reasoning layers were added: overall and per patient.

What reasoning provides and does not provide was also seen. It provides error detection,
subgroup scrutiny and a concrete object a clinician can discuss. It does not provide
causation or an assurance of correctness.

---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The dataset
you used is an open collection prepared for teaching and does not represent the patient
population of your own institution. The material is for teaching.
